<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/3_1-BART-DRS-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is a template that contains the standard setup.
It cshould be duplicated and renamed for other notebooks.

# Setup

In [ ]:
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

Conda packages

In [ ]:
import matplotlib
import numpy
import pandas
import scipy
import sklearn

Project sourcecode packages

In [ ]:
from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.sheet_tools import get_sheet_dfs

Data

In [ ]:
bart_drs_predictions_df  = pandas.read_csv(os.path.join(output_path, 'BART_DRS_results.csv'))

In [ ]:
drs_col = 'Current Digital Readiness Score (refer to PAS:1040)'

In [ ]:
col_types = {
    'Client ID': 'str',
    'Current Digital Readiness Score (refer to PAS:1040)': pandas.Int64Dtype(),
    'Number of GAFs': 'int',
    'Predicted DRS': 'int',
    'Probability': 'float',
    'Confidence': 'float'
}

bart_drs_predictions_df[drs_col] = bart_drs_predictions_df[drs_col].replace(to_replace='', value=numpy.nan)

for col, ty in col_types.items():
  bart_drs_predictions_df[col] = bart_drs_predictions_df[col].astype(ty)

# Analysis

In [ ]:
bart_drs_predictions_df[drs_col].value_counts().sort_index()

In [ ]:
bart_drs_predictions_df['Predicted DRS'].value_counts().sort_index()

In [ ]:
x = numpy.arange(1, len(DRS_LEVELS)+1)
width = 1/3

fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))

multiplier = -1

for col, label in {'Current Digital Readiness Score (refer to PAS:1040)': 'Expert-assigned', 'Predicted DRS': 'BART-predicted'}.items():
  y = bart_drs_predictions_df[col].value_counts()
  y = y.reindex(range(1, len(DRS_LEVELS)+1), fill_value=0)
  offset = width * multiplier
  rects = ax.bar(
      x + offset,
      y,
      width,
      label=label
  )
  multiplier += 1

ax.set_xticks(x)

ax.legend()

ax.set_ylabel('Number of firms')
ax.set_xlabel('Digital Readiness Score')

# Direct comparison

In [ ]:
bart_drs_predictions_df[[drs_col, 'Predicted DRS']].corr()

In [ ]:
def calculate_accuracy(df, max_diff):
  diffs = (df[drs_col] - df['Predicted DRS']).abs()
  return (diffs <= max_diff).mean(skipna=True)

print(f'Accuracy: {calculate_accuracy(bart_drs_predictions_df, 0)}')
print(f'Adjacent accuracy: {calculate_accuracy(bart_drs_predictions_df, 1)}')

In [ ]:
comparison_df = bart_drs_predictions_df.groupby([drs_col, 'Predicted DRS']).size().reset_index(name='count')
heat = numpy.zeros((9,9))
heat[
      comparison_df[drs_col].to_numpy()-1,
      comparison_df['Predicted DRS'].to_numpy()-1
    ] = comparison_df['count'].to_numpy().T
heat = heat.T
heat

In [ ]:
lr_model = sklearn.linear_model.LinearRegression()
# Check reshaping
X = bart_drs_predictions_df.dropna()['Current Digital Readiness Score (refer to PAS:1040)'].to_numpy().reshape(-1, 1)
Y = bart_drs_predictions_df.dropna()['Predicted DRS'].to_numpy()
lr_model.fit(X, Y)

In [ ]:
lr_model.score(X, Y)

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))

im = ax.imshow(heat, cmap="Reds")

ax.xaxis.set_inverted(False)
ax.yaxis.set_inverted(False)

ax.set_xlabel('Expert-assigned DRS')
ax.set_ylabel('BART-predicted DRS')

ax.set_xticks(range(0,9), labels=[str(i) for i in range(1,10)])
ax.set_yticks(range(0,9), labels=[str(i) for i in range(1,10)])

cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046)
cbar.ax.set_ylabel("Number\nof firms", rotation=0, y=1.01, labelpad=3)

In [ ]:
bart_drs_predictions_df['Prediction diff'] = bart_drs_predictions_df[drs_col] - bart_drs_predictions_df['Predicted DRS']
pred_diff_counts = bart_drs_predictions_df.groupby('Prediction diff').size().reset_index(name='count')
pred_diff_counts

In [ ]:
bart_drs_predictions_df['Pred 5 diff'] = bart_drs_predictions_df[drs_col] - 5
pred_5_diff_counts = bart_drs_predictions_df.groupby('Pred 5 diff').size().reset_index(name='count')
pred_5_diff_counts

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))

ax.bar(pred_diff_counts['Prediction diff'], pred_diff_counts['count'])
ax.plot(pred_5_diff_counts['Pred 5 diff'], pred_5_diff_counts['count'], c='r')


ax.set_xlabel('Prediction diff')
ax.set_ylabel('Count')


In [ ]:
bart_drs_predictions_df['Prediction diff'].mean()

In [ ]:
bart_drs_predictions_df['Prediction diff'].std()

In [ ]:
bart_drs_predictions_df['Prediction diff'].plot.density()

In [ ]:
x = numpy.linspace(bart_drs_predictions_df['Prediction diff'].min(), bart_drs_predictions_df['Prediction diff'].max(), 100)
pred_diff_norm_pdf = scipy.stats.norm.pdf(
    x,
    loc=bart_drs_predictions_df['Prediction diff'].mean(),
    scale=bart_drs_predictions_df['Prediction diff'].std()
  )
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.plot(x, pred_diff_norm_pdf)